# Эксперимент Дениса (dense)
Повторяем обычный (dense) запуск из ноутбука Дениса, используя только модули transformer_lab.

Отличия от Дениса, принятые сознательно:
- токенизатор GPT-2 BPE (50 257) вместо SentencePiece-16K, поэтому параметров 66.6M вместо 49.1M, и perplexity с ним не сравнима;
- корпус — один поток без `eos` между статьями;
- нет grad accumulation: batch 8 вместо 2×4, токенов на шаг столько же (4096).

Устанавливаем наш пакет

In [ ]:
%pip install -q git+https://github.com/tereshkovyv/transformer_lab.git

Подключаем Гугл диск, чтобы хранить там данные и результаты

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/transformer_lab/data/wikitext-103"
RUN_DIR = "/content/drive/MyDrive/transformer_lab/runs/denis_dense_seed44"

Настраиваем конфигурацию (как у Дениса в режиме `colab`)

`context_length` = 2048 задаёт только размер буферов RoPE и маски (для бенчмарков до 2048), обучаем на последовательностях длины `max_length` = 512.

In [ ]:
SEED = 44
cfg = {"vocab_size": 50257, "context_length": 2048, "emb_dim": 512, "n_heads": 8,
       "n_layers": 12, "hidden_dim": 1536, "drop_rate": 0.0, "qkv_bias": False}
hp = {"max_length": 512, "batch_size": 8, "lr": 3e-4, "weight_decay": 0.1, "betas": (0.9, 0.95),
      "warmup_steps": 102, "min_lr": 3e-5, "train_tokens_limit": 20_000_000, "train_steps": 2048,
      "eval_freq": 250, "eval_iter": 8, "use_amp": True}

Создаем объекты данных

У Дениса train обрезан до первых 20M токенов, и 2048 шагов × 8 последовательностей берутся из них в случайном порядке. Повторяем это через `Subset`: случайные 16 384 последовательности из первых 39 062 — одна эпоха = 2048 шагов = 8.39M токенов.

In [ ]:
import torch
from torch.utils.data import DataLoader, Subset
from transformer_lab.training.data import TokenDataset

device = "cuda"
torch.manual_seed(SEED)

train_ds = TokenDataset("train", hp["max_length"], DATA_DIR)
val_ds = TokenDataset("validation", hp["max_length"], DATA_DIR)

n_pool = hp["train_tokens_limit"] // hp["max_length"]
g = torch.Generator().manual_seed(SEED)
train_idx = torch.randperm(n_pool, generator=g)[: hp["train_steps"] * hp["batch_size"]]
train_loader = DataLoader(Subset(train_ds, train_idx.tolist()), batch_size=hp["batch_size"],
                          shuffle=False, drop_last=True)  # order is already random
val_loader = DataLoader(val_ds, batch_size=hp["batch_size"], shuffle=False, drop_last=False)
print("train steps:", len(train_loader), "val batches:", len(val_loader))

Создаем модель и оптимайзер

In [ ]:
from transformer_lab.model.gpt_model import GPTModelV2

model = GPTModelV2(cfg).to(device)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")
optimizer = torch.optim.AdamW(model.parameters(), lr=hp["lr"],
                              weight_decay=hp["weight_decay"], betas=hp["betas"])

Обучаем: warmup + cosine + gradient clipping (Raschka, приложение D) и fp16 AMP

In [ ]:
import time
import tiktoken
from transformer_lab.training.train import train_model

tokenizer = tiktoken.get_encoding("gpt2")
start = time.perf_counter()
train_losses, val_losses, tokens_seen, lrs = train_model(
    model, train_loader, val_loader, optimizer, device,
    n_epochs=1, eval_freq=hp["eval_freq"], eval_iter=hp["eval_iter"],
    start_context="The", tokenizer=tokenizer,
    warmup_steps=hp["warmup_steps"], initial_lr=hp["lr"] / hp["warmup_steps"],
    min_lr=hp["min_lr"], use_amp=hp["use_amp"])
train_seconds = time.perf_counter() - start
print(f"training: {train_seconds / 60:.1f} min")

Считаем loss и perplexity на всём validation, генерируем образец как у Дениса (30 токенов от "The")

In [ ]:
import math
from transformer_lab.model.generate import generate_text_simple
from transformer_lab.training.train import calc_loss_loader

model.eval()
with torch.no_grad():
    val_loss = calc_loss_loader(val_loader, model, device)
print(f"val loss {val_loss:.4f}, perplexity {math.exp(val_loss):.2f}")

encoded = torch.tensor(tokenizer.encode("The")).unsqueeze(0).to(device)
sample = tokenizer.decode(generate_text_simple(model, encoded, max_new_tokens=30,
                                               context_size=hp["max_length"]).squeeze(0).tolist())
print(sample)

Рисуем кривые обучения

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(tokens_seen, train_losses, label="train")
axes[0].plot(tokens_seen, val_losses, label="validation")
axes[0].set(title="Loss", xlabel="Processed tokens", ylabel="Loss")
axes[0].legend()
axes[1].plot(lrs)
axes[1].set(title="Learning rate", xlabel="Step", ylabel="lr")
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

Сохраняем веса, конфиг и результаты на диск

In [ ]:
import json
import os

os.makedirs(RUN_DIR, exist_ok=True)
torch.save(model.state_dict(), f"{RUN_DIR}/model.pth")
fig.savefig(f"{RUN_DIR}/loss.png", dpi=180, bbox_inches="tight")
with open(f"{RUN_DIR}/results.json", "w", encoding="utf-8") as f:
    json.dump({
        "seed": SEED, "cfg": cfg, "hp": hp,
        "train_losses": train_losses, "val_losses": val_losses,
        "tokens_seen": tokens_seen, "lrs": lrs,
        "final_val_loss": val_loss, "final_perplexity": math.exp(val_loss),
        "sample": sample, "train_seconds": train_seconds,
        "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0),
    }, f, ensure_ascii=False, indent=2)